In [2]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [3]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


In [4]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 124 files in the knowledge base
Total characters in knowledge base: 278,192


In [5]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 71,376


In [6]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 124 documents


In [10]:
documents[30]

Document(metadata={'source': 'knowledge-base\\flights_routes\\india_bangkok.md', 'doc_type': 'flights_routes'}, page_content='# SkyNest Route: India (Mumbai BOM) → Bangkok Suvarnabhumi (BKK)\n\n---\n\n## Route Summary\n\n| Detail | Info |\n|---|---|\n| **Origin** | India (Mumbai BOM) |\n| **Destination** | Bangkok Suvarnabhumi (BKK) |\n| **Typical Duration** | 3h 30m |\n| **Economy Fares from** | $210 |\n| **Business Class from** | $720 |\n\n---\n\n## Flight Schedule & Pricing\n\n### SN109 — Mumbai (BOM) → Bangkok (BKK)\n- **Departure**: 10:45 | **Arrival**: 16:15\n- **Duration**: 3h 30m\n- **Aircraft**: Airbus A320\n- **Frequency**: Daily\n- **Economy**: $210\n- **Business Class**: $720\n\n---\n\n## In-Flight Amenities\n\nWi-Fi, snacks (Economy), hot meal (Business), entertainment screen.\n\n---\n\n## Route Notes & Tips\n\nPopular year-round. Indian passport holders can obtain a Visa-on-Arrival at Bangkok Suvarnabhumi Airport (cost: THB 2,000 / ~$55, valid 15 days). Peak: Dec–Feb, Jul

In [8]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 398 chunks
First chunk:

page_content='# Car Rental Fleet — Bangkok (Thailand) | SkyNest Partner Cars

**Partner**: SkyRent Thailand + LuxuryCars Thailand  
**Important**: Left-hand traffic. Thailand driving licence or IDP required. Traffic in Bangkok is extremely heavy — do NOT drive in the city unless necessary. Grab is strongly recommended for city travel. Cars are ideal for day trips to Ayutthaya, Kanchanaburi, Pattaya, or overland to Chiang Mai. Fuel (benzene/gasohol) is affordable (~THB 47/litre = ~$1.30/litre).

---

## Budget Category — Economy / Basic (most affordable)

### Honda Jazz
- **Daily Rate**: $22/day
- **Seats**: 5
- **Features**: AC, Bluetooth, fuel-efficient, city-friendly
- **Best For**: Airport transfer, Pattaya trip, budget city use

### Toyota Yaris
- **Daily Rate**: $20/day
- **Seats**: 5
- **Features**: AC, reliable, low running cost
- **Best For**: Budget travellers, solo, couple

## Standard Category — Mid-range (comfort + features)' metadata={

In [11]:
chunks[100]

Document(metadata={'source': 'knowledge-base\\flights_policies\\meals_loyalty.md', 'doc_type': 'flights_policies'}, page_content='---\n\n### Meal Categories Available on Request\n\nAll meal requests must be placed **at least 24 hours before departure** via the SkyNest app, website, or by calling Customer Care.\n\n| Meal Code | Description |\n|---|---|\n| VGML | Vegan Meal — no animal products |\n| AVML | Asian Vegetarian Meal — Indian vegetarian (no eggs) |\n| VJML | Jain Meal — strict vegetarian, no root vegetables |\n| HNML | Hindu Non-Vegetarian Meal — halal not guaranteed |\n| MOML | Muslim Meal — Halal certified throughout |\n| KSML | Kosher Meal — sealed, certified by rabbinical authority |\n| BLML | Bland Meal — low spice, easy on digestion |\n| DBML | Diabetic Meal — low sugar, controlled carbohydrates |\n| GFML | Gluten-Free Meal |\n| LCML | Low-Calorie Meal — < 600 kcal per meal |\n| LFML | Low-Fat Meal |\n| LSML | Low-Sodium Meal |\n| PRML | Low-Protein Meal (renal-friendly)

### PART B: Make vectors and store in Chroma

In [12]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\jeffy\projects\llm_engineering\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jeffy\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectorstore created with 398 documents


In [13]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 398 vectors with 384 dimensions in the vector store


In [14]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]

color_map = {
    'cars':           'red',
    'company':        'blue',
    'destinations':   'green',
    'flights_policies': 'orange',
    'flights_routes': 'purple',
    'hotels':         'brown',
    'packages':       'pink',
    'visa_policies':  'cyan',
}

colors = [color_map.get(t, 'gray') for t in doc_types]

In [15]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()